In [1]:
import os
import json
import glob
import xml.etree.ElementTree as ET
from datasets import load_from_disk

d:\2B_proj\Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
print(os.getcwd())

d:\2B_proj\Project\notebooks


In [3]:
from datasets import load_from_disk

ds = load_from_disk("../data/raw/meddialog_en")
print(ds)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 112165
})


In [4]:
print(ds.column_names)

['instruction', 'input', 'output']


In [5]:
print(ds[0])        # first sample

{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea

In [6]:
print(ds[:5])       # first 5 samples

{'instruction': ["If you are a doctor, please answer the medical questions based on the patient's description.", "If you are a doctor, please answer the medical questions based on the patient's description.", "If you are a doctor, please answer the medical questions based on the patient's description.", "If you are a doctor, please answer the medical questions based on the patient's description.", "If you are a doctor, please answer the medical questions based on the patient's description."], 'input': ['I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself,

In [7]:
import pandas as pd

df = ds.to_pandas()
df.head()

,instruction,input,output
0,"If you are a doctor, please answer the medical...",I woke up this morning feeling the whole room ...,"Hi, Thank you for posting your query. The most..."
1,"If you are a doctor, please answer the medical...",My baby has been pooing 5-6 times a day for a ...,Hi... Thank you for consulting in Chat Doctor....
2,"If you are a doctor, please answer the medical...","Hello, My husband is taking Oxycodone due to a...","Hello, and I hope I can help you today.First, ..."
3,"If you are a doctor, please answer the medical...",lump under left nipple and stomach pain (male)...,HI. You have two different problems. The lump ...
4,"If you are a doctor, please answer the medical...",I have a 5 month old baby who is very congeste...,Thank you for using Chat Doctor. I would sugge...


In [8]:
df['instruction'].nunique()

1

In [9]:
df['input'].nunique()

112003

In [10]:
df['output'].nunique()

109310

MedQuad

In [16]:
xml_files = glob.glob("../data/raw/MedQuAD/**/*.xml", recursive=True)
print(f"Total XML files: {len(xml_files)}")

# Look at first valid QA pair
for xml_file in xml_files:
    root = ET.parse(xml_file).getroot()
    for qa in root.findall(".//QAPair"):
        q = qa.find("Question")
        a = qa.find("Answer")
        if a is not None and a.text and len(a.text.strip()) > 20:
            print(f"\nQ: {q.text[:200]}")
            print(f"A: {a.text[:200]}")
            break
    else:
        continue
    break

Total XML files: 11274

Q: What is (are) Adult Acute Lymphoblastic Leukemia ?
A: Key Points
                    - Adult acute lymphoblastic leukemia (ALL) is a type of cancer in which the bone marrow makes too many lymphocytes (a type of white blood cell).    - Leukemia may affect


MedDialog

In [11]:
ds = load_from_disk("../data/raw/meddialog_en")
print(f"Total examples: {len(ds)}")
print(f"Columns: {ds.column_names}")
print("\nExample:")
print(f"Q: {ds[0]['input']}")
print(f"A: {ds[0]['output']}")

Total examples: 112165
Columns: ['instruction', 'input', 'output']

Example:
Q: I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!
A: Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomiting are common. The condition is due to pr

In [ ]:
# Show in nanochat format
q = ds[0]['input'].strip()
a = ds[0]['output'].strip()

conversation = [
    {"role": "user",      "content": q},
    {"role": "assistant", "content": a}
]

print(json.dumps(conversation, indent=2))

[
  {
    "role": "user",
    "content": "I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!"
  },
  {
    "role": "assistant",
    "content": "Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse with movements. Accompanying nausea and vomiting are common. The condition is

In [14]:
for split in ["train", "val", "test"]:
    path = f"../data/splits/{split}.jsonl"
    with open(path) as f:
        n = sum(1 for _ in f)
    print(f"{split}: {n:,} examples")

train: 109,277 examples
val: 12,856 examples
test: 6,429 examples


In [17]:
total = 0
has_answer = 0
no_answer = 0

for xml_file in xml_files:
    root = ET.parse(xml_file).getroot()
    for qa in root.findall(".//QAPair"):
        a = qa.find("Answer")
        total += 1
        if a is not None and a.text and len(a.text.strip()) > 20:
            has_answer += 1
        else:
            no_answer += 1

print(f"Total QA pairs : {total:,}")
print(f"Has answer     : {has_answer:,}")
print(f"No answer      : {no_answer:,}")

Total QA pairs : 47,441
Has answer     : 16,406
No answer      : 31,035


p2

In [1]:
import sys
import os
import json

sys.path.insert(0, r"D:\2B_proj\nanochat")

from nanochat.tokenizer import get_tokenizer

tok = get_tokenizer()

special_tokens = [
    "<|bos|>",
    "<|user_start|>",
    "<|user_end|>",
    "<|assistant_start|>",
    "<|assistant_end|>"
]

print("Special tokens:")
for token in special_tokens:
    tid = tok.encode_special(token)
    print(f"  {'[OK]' if tid else '✗'} {token:25s} → id: {tid}")

print(f"\nVocab size: {tok.get_vocab_size():,}")

Special tokens:
  [OK] <|bos|>                   → id: 32759
  [OK] <|user_start|>            → id: 32760
  [OK] <|user_end|>              → id: 32761
  [OK] <|assistant_start|>       → id: 32762
  [OK] <|assistant_end|>         → id: 32763

Vocab size: 32,768


In [3]:
medical_terms = [
    "dehydration",
    "tachycardia", 
    "acetaminophen",
    "lymphoblastic",
    "hypertension",
    "antibiotics",
]

print("Medical term tokenization:")
for term in medical_terms:
    tokens = tok.encode(term)
    status = "✓ single" if len(tokens) == 1 else f"→ {len(tokens)} pieces"
    print(f"  {status}  '{term}'")

Medical term tokenization:
  → 2 pieces  'dehydration'
  → 2 pieces  'tachycardia'
  ✓ single  'acetaminophen'
  → 2 pieces  'lymphoblastic'
  ✓ single  'hypertension'
  ✓ single  'antibiotics'


In [4]:
print("How terms split:")
for term in medical_terms:
    tokens = tok.encode(term)
    pieces = [tok.decode([t]) for t in tokens]
    print(f"  '{term}' → {pieces}")

How terms split:
  'dehydration' → ['de', 'hydration']
  'tachycardia' → ['t', 'achycardia']
  'acetaminophen' → ['acetaminophen']
  'lymphoblastic' → ['lymph', 'oblastic']
  'hypertension' → ['hypertension']
  'antibiotics' → ['antibiotics']


In [5]:
import torch
import pickle

tokenizer_dir = r"C:\Users\user\.cache\nanochat\tokenizer"

# token_bytes.pt 
token_bytes = torch.load(tokenizer_dir + r"\token_bytes.pt")
print(f"token_bytes.pt:")
print(f"Shape: {token_bytes.shape}")
print(f"Min bytes: {token_bytes.min().item()}")
print(f"Max bytes: {token_bytes.max().item()}")
print(f"Mean bytes: {token_bytes.float().mean().item():.2f}")

#  tokenizer.pkl 
with open(tokenizer_dir + r"\tokenizer.pkl", "rb") as f:
    enc = pickle.load(f)

print(f"\ntokenizer.pkl:")
print(f"Type: {type(enc)}")
print(f"Vocab size: {enc.n_vocab}")
print(f"Special tokens: {enc.special_tokens_set}")

token_bytes.pt:
Shape: torch.Size([32768])
Min bytes: 0
Max bytes: 32
Mean bytes: 6.71

tokenizer.pkl:
Type: <class 'tiktoken.core.Encoding'>
Vocab size: 32768
Special tokens: {'<|python_start|>', '<|python_end|>', '<|user_end|>', '<|assistant_end|>', '<|assistant_start|>', '<|output_end|>', '<|bos|>', '<|user_start|>', '<|output_start|>'}


In [6]:
import pickle
from nanochat.tokenizer import RustBPETokenizer

with open(r"C:\Users\user\.cache\nanochat\tokenizer\tokenizer.pkl", "rb") as f:
    enc = pickle.load(f)

tok = RustBPETokenizer(enc, "<|bos|>")

sentences = [
    "What are the symptoms of dehydration?",
    "The patient has acute lymphoblastic leukemia.",
    "Take 500mg of acetaminophen every 6 hours.",
    "I have a fever and sore throat.",
    "Benign paroxysmal positional vertigo causes dizziness.",
    "What is ulcer?"
]

for sentence in sentences:
    tokens = tok.encode(sentence)
    pieces = [tok.decode([t]) for t in tokens]
    print(f"\n'{sentence}'")
    print(f"  Pieces: {pieces}")
    print(f"  Count:  {len(tokens)} tokens")


'What are the symptoms of dehydration?'
  Pieces: ['What', ' are', ' the', ' symptoms', ' of', ' dehydration', '?']
  Count:  7 tokens

'The patient has acute lymphoblastic leukemia.'
  Pieces: ['The', ' patient', ' has', ' acute', ' lymphoblastic', ' leukemia', '.']
  Count:  7 tokens

'Take 500mg of acetaminophen every 6 hours.'
  Pieces: ['Take', ' ', '50', '0', 'mg', ' of', ' acetaminophen', ' every', ' ', '6', ' hours', '.']
  Count:  12 tokens

'I have a fever and sore throat.'
  Pieces: ['I', ' have', ' a', ' fever', ' and', ' sore', ' throat', '.']
  Count:  8 tokens

'Benign paroxysmal positional vertigo causes dizziness.'
  Pieces: ['Benign', ' paroxysmal', ' positional', ' vertigo', ' causes', ' dizziness', '.']
  Count:  7 tokens

'What is ulcer?'
  Pieces: ['What', ' is', ' ulcer', '?']
  Count:  4 tokens


- The tokenizer handles medical terminology very well. 
- The number splitting is nanochat's deliberate choice to save vocab space for more useful tokens.

In [71]:
empty_files = {}
MEDQUAD_DIR = r'D:\2B_proj\Project\data\raw\MedQuAD'
for xml_file in glob.glob(MEDQUAD_DIR + "/**/*.xml", recursive=True):
    root = ET.parse(xml_file).getroot()
    total = 0
    empty = 0
    for qa in root.findall(".//QAPair"):
        a = qa.find("Answer")
        total += 1
        if a is None or not a.text or len(a.text.strip()) <= 20:
            empty += 1
    if empty > 0:
        folder = xml_file.split("\\")[-2]
        empty_files[folder] = empty_files.get(folder, 0) + empty

print("Folders with empty answers:")
for folder, count in sorted(empty_files.items(), key=lambda x: -x[1]):
    print(f"  {folder:50s} → {count} empty")

Folders with empty answers:
  10_MPlus_ADAM_QA                                   → 17348 empty
  11_MPlusDrugs_QA                                   → 12889 empty
  12_MPlusHerbsSupplements_QA                        → 792 empty
  2_GARD_QA                                          → 5 empty
  9_CDC_QA                                           → 1 empty


In [60]:
import string

print(f"{'ID':>6}  {'Bytes':>5}  Token")
print("-" * 40)

for token_id in range(32768):
    token_str = tok.decode([token_id])
    byte_size = token_bytes[token_id].item()
    # Only show tokens 8+ bytes that contain actual letters
    if byte_size >= 20 and any(c.isalpha() for c in token_str):
        print(f"{token_id:>6}  {byte_size:>5}  {repr(token_str)}")

    ID  Bytes  Token
----------------------------------------
 13467     20  ' hyperparathyroidism'
 13841     20  ' bronchoconstriction'
 15265     20  ' leukoencephalopathy'
 20533     20  ' electrophysiologist'
 22562     20  ' lymphohistiocytosis'
 23644     21  ' hypercholesterolemia'
 23970     20  ' Leukoencephalopathy'
 24679     22  ' mucopolysaccharidosis'
 26566     22  ' hysterosalpingography'
 27258     22  ' Mucopolysaccharidosis'
 28449     21  ' adrenoleukodystrophy'
 29149     20  ' Hyperparathyroidism'
 29727     21  ' electroencephalogram'
 29964     20  ' histosalpingography'
 29977     20  ' postcholecystectomy'
 30501     20  ' lymphoproliferative'
 31607     20  ' neuroacanthocytosis'
 31653     27  ' hepaticopancreaticobiliary'
 31714     20  ' Spondylometaphyseal'


In [61]:
import torch
import sys
sys.path.insert(0, r"D:\2B_proj\Project")

ckpt = torch.load(r"D:\2B_proj\Project\checkpoints\smoke_test.pt", map_location="cpu")
print("Keys:", ckpt.keys())
print("Layers:")
for k, v in ckpt["model_state_dict"].items():
    print(f"  {k:50s} {str(v.shape)}")

Keys: dict_keys(['model_state_dict'])
Layers:
  token_embedding_table.weight                       torch.Size([32768, 512])
  position_embedding_table.weight                    torch.Size([64, 512])
  blocks.0.layer_norm1.weight                        torch.Size([512])
  blocks.0.layer_norm1.bias                          torch.Size([512])
  blocks.0.self_att.key.weight                       torch.Size([512, 512])
  blocks.0.self_att.query.weight                     torch.Size([512, 512])
  blocks.0.self_att.value.weight                     torch.Size([512, 512])
  blocks.0.self_att.proj.weight                      torch.Size([512, 512])
  blocks.0.self_att.proj.bias                        torch.Size([512])
  blocks.0.layer_norm2.weight                        torch.Size([512])
  blocks.0.layer_norm2.bias                          torch.Size([512])
  blocks.0.feed_forward.net.0.weight                 torch.Size([2048, 512])
  blocks.0.feed_forward.net.0.bias                   torch.Size([

In [8]:
sys.path.insert(0, r"D:\2B_proj\Project")
import config as config
from chat_model.models.model import NanoChat

print("Building model")
model = NanoChat(config=config)
params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {params:,}")
print(f"  VOCAB_SIZE: {config.VOCAB_SIZE}")
print(f"  BLOCK_SIZE: {config.BLOCK_SIZE}")

print("Forward pass")
x = torch.randint(0, config.VOCAB_SIZE, (2, config.BLOCK_SIZE))
y = torch.randint(0, config.VOCAB_SIZE, (2, config.BLOCK_SIZE))
logits, loss = model(x, y)
print(f"  Input:  {x.shape}")
print(f"  Output: {logits.shape}")
print(f"  Loss:   {loss.item():.4f}")

print("Generate test")
prompt = torch.randint(0, config.VOCAB_SIZE, (1, 5))
output = model.generate(prompt, max_new_tokens=10)
print(f"  Prompt tokens: {prompt.shape[1]}")
print(f"  Output tokens: {output.shape[1]}")

ModuleNotFoundError: No module named 'chat_model.models.model'

In [2]:
import os
import json
import pickle

with open(r"C:\Users\user\.cache\nanochat\tokenizer\tokenizer.pkl", "rb") as f:
    enc = pickle.load(f)

mergeable = {
    k.decode("utf-8", errors="replace"): v
    for k, v in enc._mergeable_ranks.items()
}

special = enc._special_tokens

vocab = {**mergeable, **special}

save_path = r"..\data\tokens\tokens.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print(f"Saved vocab to {save_path}")

Saved vocab to ..\data\tokens\tokens.json
